# 01. Oxford-IIIT Pet 데이터 분리

## 이번 노트북의 목표

Oxford-IIIT Pet Dataset을 강아지/고양이 이진 분류 문제에 맞게 바꾸고, `train`, `validation`, `test` 세 가지 데이터셋으로 직접 나눈다.

이번 단계에서 보여주고 싶은 것은 단순히 코드를 돌린 것이 아니라, **왜 이 데이터셋을 골랐고, 왜 이렇게 나눴는지 판단하는 과정**이다.

## 1. 지금 결정해야 할 질문

**질문:** 왜 Kaggle 데이터셋 대신 Oxford-IIIT Pet Dataset을 사용할까?

**선택지:**

- Kaggle Cat and Dog: train/test가 이미 나뉘어 있어서 편하다.
- Oxford-IIIT Pet: 직접 dog/cat 라벨을 만들고 train/validation/test를 나눠야 한다.

**내 선택:** Oxford-IIIT Pet Dataset을 사용한다.

**이유:** 이번 과제의 핵심 중 하나가 train/validation/test를 직접 구성하고 각 데이터셋의 역할을 설명하는 것이기 때문이다. 이미 나뉜 데이터를 쓰기보다 직접 나누는 편이 학습 과정과 발표에서 더 설득력 있다.

## 2. train / validation / test의 역할

초보자 관점에서는 세 데이터를 이렇게 이해한다.

- **train set:** 모델이 공부하는 문제집
- **validation set:** 학습 중 선택을 점검하는 모의고사
- **test set:** 마지막에 한 번만 보는 최종 시험

중요한 점은 test set을 학습 중에 자주 보면 안 된다는 것이다. test set은 모델을 다 만든 뒤 마지막 성능 확인에 사용해야 한다.

## 3. 필요한 라이브러리 불러오기

이번 노트북에서는 데이터셋 로드, 표 정리, 데이터 분리, 그래프 저장을 한다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision.datasets import OxfordIIITPet

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(PROJECT_ROOT)

## 4. 데이터셋 로드

Oxford-IIIT Pet Dataset은 37개 품종으로 구성되어 있다. 우리는 품종 분류가 아니라 dog/cat 이진 분류를 할 것이므로, 먼저 원본 데이터에서 이미지 경로와 품종 이름을 가져온다.

처음 실행할 때는 `download=True` 때문에 데이터 다운로드가 진행된다.

In [ ]:
dataset = OxfordIIITPet(
    root=RAW_DIR,
    split="trainval",
    target_types="category",
    download=True,
)

print(f"number of images: {len(dataset)}")
print(f"number of classes: {len(dataset.classes)}")
print(dataset.classes[:5])

## 5. dog/cat 라벨 변환 규칙

Oxford-IIIT Pet의 클래스 이름은 품종 이름이다. 이 데이터셋에서는 일반적으로 **고양이 품종 이름이 대문자로 시작**하고, **강아지 품종 이름이 소문자로 시작**한다.

예시:

- `Abyssinian`, `Bengal`, `Persian` → cat
- `american_bulldog`, `beagle`, `pug` → dog

따라서 클래스 이름의 첫 글자가 대문자인지 확인해서 이진 라벨을 만든다.

In [ ]:
def breed_to_binary_label(breed_name: str) -> str:
    """Convert Oxford-IIIT Pet breed name to cat/dog label."""
    return "cat" if breed_name[0].isupper() else "dog"


records = []
for image_path, class_index in zip(dataset._images, dataset._labels):
    breed_name = dataset.classes[class_index]
    binary_label = breed_to_binary_label(breed_name)
    records.append(
        {
            "image_path": str(image_path),
            "breed": breed_name,
            "label": binary_label,
        }
    )

df = pd.DataFrame(records)
df.head()

## 6. 원본 라벨 분포 확인

데이터를 나누기 전에 먼저 전체 데이터에서 cat/dog 개수가 어느 정도인지 확인한다. 만약 한쪽 클래스가 너무 적다면 accuracy만으로 성능을 판단하기 어려워질 수 있다.

In [ ]:
overall_distribution = df["label"].value_counts().rename_axis("label").reset_index(name="count")
overall_distribution

## 7. split 비율 결정

**질문:** train/validation/test를 어떤 비율로 나눌까?

**선택지:**

- 80/10/10: train이 많아서 학습에는 유리하지만 validation/test가 작다.
- 70/15/15: 학습 데이터도 충분히 남기고, 검증/평가 데이터도 어느 정도 확보한다.
- 60/20/20: 검증/평가는 넉넉하지만 학습 데이터가 줄어든다.

**내 선택:** 70/15/15

**이유:** 이번 과제는 성능 최고점보다 split의 역할을 설명하고 평가를 해석하는 것이 중요하다. validation과 test를 너무 작게 두지 않는 편이 결과 해석에 좋다.

## 8. stratified split 생성

`stratify`를 사용해 각 split에 dog/cat 비율이 비슷하게 들어가도록 나눈다. 이렇게 해야 특정 split에 한 클래스가 몰리는 문제를 줄일 수 있다.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    train_size=TRAIN_RATIO,
    random_state=RANDOM_SEED,
    stratify=df["label"],
)

val_fraction_of_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df,
    train_size=val_fraction_of_temp,
    random_state=RANDOM_SEED,
    stratify=temp_df["label"],
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
split_df = split_df[["split", "label", "breed", "image_path"]]
split_df.head()

## 9. split 결과 저장

이후 학습 노트북에서는 같은 split을 그대로 사용해야 한다. 그래서 split 결과를 CSV로 저장한다. 이렇게 하면 매번 데이터가 다르게 나뉘는 문제를 피할 수 있다.

In [ ]:
split_path = PROCESSED_DIR / "pet_binary_splits.csv"
split_df.to_csv(split_path, index=False)
print(split_path)

## 10. split별 dog/cat 분포 확인

잘 나뉘었는지 보기 위해 split별 dog/cat 개수를 표로 확인한다. 여기서 중요한 것은 세 split 모두에 dog와 cat이 비슷한 비율로 들어갔는지 보는 것이다.

In [ ]:
distribution = (
    split_df.groupby(["split", "label"])
    .size()
    .rename("count")
    .reset_index()
)

distribution_path = PROCESSED_DIR / "class_distribution.csv"
distribution.to_csv(distribution_path, index=False)
distribution

In [ ]:
pivot_distribution = distribution.pivot(index="split", columns="label", values="count")

ax = pivot_distribution.loc[["train", "validation", "test"]].plot(
    kind="bar",
    figsize=(8, 5),
    rot=0,
    color=["#4C78A8", "#F58518"],
)
ax.set_title("Dog/Cat Count by Split")
ax.set_xlabel("split")
ax.set_ylabel("count")
ax.legend(title="label")
plt.tight_layout()

figure_path = FIGURE_DIR / "class_distribution.png"
plt.savefig(figure_path, dpi=150)
plt.show()

print(figure_path)

## 11. 결과를 보고 판단하기

확인할 점:

1. train, validation, test가 의도한 비율에 가깝게 나뉘었는가?
2. 각 split 안에 dog/cat이 모두 들어 있는가?
3. 특정 split에 한 클래스가 지나치게 몰리지 않았는가?

이 조건이 만족되면 다음 학습 단계로 넘어갈 수 있다.

In [ ]:
split_summary = split_df.groupby("split").size().rename("count").reset_index()
split_summary["ratio"] = split_summary["count"] / len(split_df)
split_summary

## 12. 발표에 넣을 정리 문장

이번 과제에서는 Kaggle처럼 이미 train/test가 나뉜 데이터셋 대신 Oxford-IIIT Pet Dataset을 사용했습니다. 그 이유는 데이터 분리 과정을 직접 경험하고, train/validation/test가 학습 과정에서 각각 어떤 역할을 하는지 설명하기 위해서입니다.

저는 전체 데이터를 dog/cat 이진 라벨로 변환한 뒤, 70/15/15 비율로 train/validation/test를 구성했습니다. 이때 각 split의 dog/cat 비율이 크게 틀어지지 않도록 stratified split을 사용했습니다.

train set은 모델이 학습하는 데이터, validation set은 학습 중 모델 선택과 overfitting 여부를 확인하는 데이터, test set은 마지막 최종 성능 평가에 사용하는 데이터로 구분했습니다.

## 13. 이번 단계에서 배운 점

- 데이터셋을 고르는 기준도 과제 목적에 맞아야 한다.
- train/validation/test는 단순히 비율로 나누는 것이 아니라 역할이 다르다.
- split을 저장해두면 이후 실험을 같은 기준으로 비교할 수 있다.
- 클래스 분포를 확인해야 accuracy, precision, recall 해석이 더 신뢰할 수 있다.